# Predictive Analytics Benchmark — Hotel Bookings Cancellation

---
## Task 1

### 1. Plan

Steps to be executed in Task 1:

1. Read `dataset_description.pdf` and record variable definitions, data sources, temporal scope, and any caveats stated by the authors.
2. Load `hotel_bookings.csv` and verify shape, dtypes, and column names against the documentation.
3. Inspect every variable: its meaning (from docs), observed range, and any inconsistency with the documented description.
4. Quantify missing values; distinguish true nulls from documented "NULL" categories.
5. Detect data quality issues: impossible numeric values, duplicate rows, zero-night bookings, inconsistent categorical levels.
6. Flag leakage and methodological risk variables, citing documentation evidence.
7. Produce a structured preprocessing plan with justification for each decision.

**Source convention used throughout:**
- *[DOC]* = interpretation derived from the documentation
- *[OBS]* = interpretation inferred from observed data patterns

### 2. Risks

| # | Risk category | Specific risk | Notes |
|---|---------------|---------------|-------|
| 1 | **Data leakage** | `reservation_status` directly encodes the cancellation outcome | *[DOC]* Authors confirm it is the "last status" of a booking — a post-outcome label |
| 2 | **Data leakage** | `reservation_status_date` is the date the final status was set | *[DOC]* Post-outcome timestamp, always after cancellation or check-out |
| 3 | **Data leakage** | `deposit_type` is computed from transactions recorded up to the arrival **or cancellation** date | *[DOC]* Table 1 footnote: "payments identified… before the booking's arrival or cancellation date" — cancellation date is post-outcome |
| 4 | **Data leakage** | `assigned_room_type` is the room actually given at check-in | *[DOC]* Assigned at check-in; may differ from reserved — not known at booking time for cancelled bookings |
| 5 | **Data leakage** | `booking_changes` is counted up to "check-in or cancellation" | *[DOC]* Changes after the prediction point (day prior to arrival) are post-outcome |
| 6 | **Distributional bias** | `country` may not be correctly known until check-in | *[DOC]* "hotels may not know the correct nationality of the customer until the moment of check-in" |
| 7 | **Data quality** | `adr` contains negative values | *[OBS]* H1 min = −6.38 per documentation; physically impossible as a rate |
| 8 | **Data quality** | Extreme outliers in `adr` (H2 max = 5400) and `adults` (max = 55) | *[OBS]* Far exceed plausible hotel values |
| 9 | **Data quality** | Zero-night bookings (`stays_in_weekend_nights` + `stays_in_week_nights` = 0) | *[OBS]* Could indicate same-day cancellations or data errors |
| 10 | **Reproducibility** | Combined dataset merges two hotels with different distributions | *[DOC]* H1 (Resort, Algarve) and H2 (City, Lisbon) have structurally different cancellation rates |

### 3. Implementation

In [1]:
# ── Task 1 | Step 0: imports ─────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

print('Imports OK')

Imports OK


In [2]:
# ── Task 1 | Step 1: Load dataset & basic schema ─────────────────────────────
df = pd.read_csv('hotel_bookings.csv')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nDtypes:\n{df.dtypes.to_string()}')

Shape: 119,390 rows × 32 columns

Dtypes:
hotel                                 str
is_canceled                         int64
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                    str
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                          float64
babies                              int64
meal                                  str
country                               str
market_segment                        str
distribution_channel                  str
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                    str
assigned_room_type                    str
booking_changes                     int64
deposit_type                      

In [3]:
# ── Task 1 | Step 2: Variable catalogue — documentation vs observed ───────────
# [DOC] 31 variables described in Table 1; combined CSV adds 'hotel' column (H1/H2 label)
variable_catalogue = {
    'hotel':                         '[DOC] Hotel identifier: "Resort Hotel" (H1, Algarve) or "City Hotel" (H2, Lisbon). Not in original per-hotel files; added on merge.',
    'is_canceled':                   '[DOC] TARGET. 1=booking canceled, 0=not canceled. Sourced from Booking table (BO).',
    'lead_time':                     '[DOC] Days between booking entry date and arrival date. Computed: arrival_date − entry_date. Pre-arrival info.',
    'arrival_date_year':             '[DOC] Year of arrival. Range 2015–2017 per documentation.',
    'arrival_date_month':            '[DOC] Month of arrival as string ("January"..."December").',
    'arrival_date_week_number':      '[DOC] ISO week number of arrival date.',
    'arrival_date_day_of_month':     '[DOC] Day of month of arrival.',
    'stays_in_weekend_nights':       '[DOC] Number of Saturday/Sunday nights booked or stayed.',
    'stays_in_week_nights':          '[DOC] Number of Monday–Friday nights booked or stayed.',
    'adults':                        '[DOC] Number of adults.',
    'children':                      '[DOC] Number of children (payable + non-payable). Sourced BO+BL.',
    'babies':                        '[DOC] Number of babies.',
    'meal':                          '[DOC] Meal package: Undefined/SC, BB, HB, FB.',
    'country':                       '[DOC] Country of origin (ISO 3155-3:2013). CAUTION: nationality may not be confirmed until check-in.',
    'market_segment':                '[DOC] Market segment: Online TA, Offline TA/TO, Direct, Corporate, Groups, Complementary, Aviation.',
    'distribution_channel':          '[DOC] Booking channel: TA/TO, Direct, Corporate, GDS, Undefined.',
    'is_repeated_guest':             '[DOC] 1 if guest profile existed before this booking, else 0.',
    'previous_cancellations':        '[DOC] Count of prior cancellations by this customer profile. 0 if no profile found.',
    'previous_bookings_not_canceled':'[DOC] Count of prior completed bookings by this customer profile.',
    'reserved_room_type':            '[DOC] Room type code at time of reservation.',
    'assigned_room_type':            '[DOC] Room type actually assigned. LEAKAGE RISK: assigned at check-in, post-booking.',
    'booking_changes':               '[DOC] Changes to booking up to check-in or cancellation. LEAKAGE RISK: includes post-booking-time changes.',
    'deposit_type':                  '[DOC] Deposit status: NoDeposit, NonRefund, Refundable. LEAKAGE RISK: computed from transactions up to arrival OR cancellation date.',
    'agent':                         '[DOC] Travel agency ID. "NULL" = no agent (not a missing value per documentation).',
    'company':                       '[DOC] Company/entity ID. "NULL" = no company (not a missing value per documentation).',
    'days_in_waiting_list':          '[DOC] Days booking spent on waiting list before confirmation.',
    'customer_type':                 '[DOC] Booking type: Contract, Group, Transient, Transient-Party.',
    'adr':                           '[DOC] Average Daily Rate = total lodging revenue / total staying nights. Can be negative (data artifact).',
    'required_car_parking_spaces':   '[DOC] Number of parking spaces requested.',
    'total_of_special_requests':     '[DOC] Count of special requests (e.g. twin bed, high floor).',
    'reservation_status':            '[DOC] Last reservation status: Canceled, Check-Out, No-Show. DIRECT LEAKAGE: post-outcome label.',
    'reservation_status_date':       '[DOC] Date last status was set. DIRECT LEAKAGE: post-outcome timestamp.',
}

print(f'Documented variables: {len(variable_catalogue)}')
print(f'CSV columns:          {df.shape[1]}')
print(f'\nAll CSV columns accounted for in catalogue: {set(df.columns) == set(variable_catalogue.keys())}')

Documented variables: 32
CSV columns:          32

All CSV columns accounted for in catalogue: True


In [4]:
# ── Task 1 | Step 3: Hotel split & temporal scope ────────────────────────────
print('=== Hotel distribution ===')
print(df['hotel'].value_counts())
# [DOC] H1 = Resort Hotel (40,060 obs), H2 = City Hotel (79,330 obs)

print('\n=== Arrival year range ===')
print(df['arrival_date_year'].value_counts().sort_index())
# [DOC] Bookings due to arrive between 1 Jul 2015 and 31 Aug 2017

print('\n=== Target distribution by hotel ===')
print(df.groupby('hotel')['is_canceled'].value_counts().unstack())
print('\nCancellation rate by hotel:')
print(df.groupby('hotel')['is_canceled'].mean().round(3))

=== Hotel distribution ===
hotel
City Hotel      79330
Resort Hotel    40060
Name: count, dtype: int64

=== Arrival year range ===
arrival_date_year
2015    21996
2016    56707
2017    40687
Name: count, dtype: int64

=== Target distribution by hotel ===
is_canceled       0      1
hotel                     
City Hotel    46228  33102
Resort Hotel  28938  11122

Cancellation rate by hotel:
hotel
City Hotel     0.417
Resort Hotel   0.278
Name: is_canceled, dtype: float64


In [5]:
# ── Task 1 | Step 4: Missing value audit ─────────────────────────────────────
# [DOC] "NULL" in agent/company = not applicable, NOT missing
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(2)
missing_df  = pd.DataFrame({'missing_n': null_counts, 'missing_%': null_pct})
missing_df  = missing_df[missing_df['missing_n'] > 0].sort_values('missing_n', ascending=False)

print('=== True NaN missing values ===')
print(missing_df if len(missing_df) else 'None')

for col in ['agent', 'company']:
    n = (df[col] == 'NULL').sum()
    pct = n / len(df) * 100
    print(f'\n[DOC] "{col}" == "NULL" (= no agent/company, not missing): {n:,} rows ({pct:.1f}%)')

=== True NaN missing values ===
          missing_n  missing_%
company      112593     94.310
agent         16340     13.690
country         488      0.410
children          4      0.000

[DOC] "agent" == "NULL" (= no agent/company, not missing): 0 rows (0.0%)

[DOC] "company" == "NULL" (= no agent/company, not missing): 0 rows (0.0%)


In [6]:
# ── Task 1 | Step 5: Data quality checks ─────────────────────────────────────
issues = {}

# 5a. Duplicate rows
n_dupes = df.duplicated().sum()
issues['duplicate_rows'] = n_dupes
print(f'Duplicate rows: {n_dupes:,}')

# 5b. Negative ADR  [DOC] H1 min = -6.38; physically impossible
neg_adr = (df['adr'] < 0).sum()
issues['negative_adr'] = neg_adr
print(f'Negative ADR rows: {neg_adr}  (min={df["adr"].min():.2f})')

# 5c. Extreme ADR outliers  [OBS]
high_adr = (df['adr'] > 1000).sum()
issues['adr_above_1000'] = high_adr
print(f'ADR > 1000: {high_adr} rows  (max={df["adr"].max():.2f})')

# 5d. Zero-night bookings  [OBS]
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
zero_nights = (df['total_nights'] == 0).sum()
issues['zero_night_bookings'] = zero_nights
print(f'Zero-night bookings: {zero_nights:,}')

# 5e. Adults = 0  [OBS]
zero_adults = (df['adults'] == 0).sum()
issues['zero_adults'] = zero_adults
print(f'Bookings with 0 adults: {zero_adults:,}')

# 5f. Extreme adult counts  [OBS]  [DOC] H1 max = 55
high_adults = (df['adults'] > 10).sum()
issues['adults_above_10'] = high_adults
print(f'Adults > 10: {high_adults}  (max={df["adults"].max()})')

# 5g. Zero total guests  [OBS]
zero_guests = ((df['adults'] + df['children'].fillna(0) + df['babies']) == 0).sum()
issues['zero_total_guests'] = zero_guests
print(f'Bookings with zero total guests: {zero_guests:,}')

# 5h. Categorical level audit  [OBS]
doc_meal_levels    = {'Undefined', 'SC', 'BB', 'HB', 'FB'}
doc_deposit_levels = {'No Deposit', 'Non Refund', 'Refundable'}
obs_meal    = set(df['meal'].unique())
obs_deposit = set(df['deposit_type'].unique())
print(f'\nMeal levels in data:    {obs_meal}')
print(f'Meal levels in docs:    {doc_meal_levels}')
print(f'Deposit levels in data: {obs_deposit}')
print(f'Deposit levels in docs: {doc_deposit_levels}')

print('\n=== Quality issue summary ===')
for k, v in issues.items():
    print(f'  {k}: {v:,}')

Duplicate rows: 31,994
Negative ADR rows: 1  (min=-6.38)
ADR > 1000: 1 rows  (max=5400.00)
Zero-night bookings: 715
Bookings with 0 adults: 403
Adults > 10: 12  (max=55)
Bookings with zero total guests: 180

Meal levels in data:    {'HB', 'Undefined', 'BB', 'SC', 'FB'}
Meal levels in docs:    {'BB', 'HB', 'SC', 'FB', 'Undefined'}
Deposit levels in data: {'Non Refund', 'No Deposit', 'Refundable'}
Deposit levels in docs: {'Non Refund', 'No Deposit', 'Refundable'}

=== Quality issue summary ===
  duplicate_rows: 31,994
  negative_adr: 1
  adr_above_1000: 1
  zero_night_bookings: 715
  zero_adults: 403
  adults_above_10: 12
  zero_total_guests: 180


In [7]:
# ── Task 1 | Step 6: Leakage confirmation — reservation_status cross-tab ─────
print('=== reservation_status x is_canceled ===')
print(pd.crosstab(df['reservation_status'], df['is_canceled'],
                  margins=True, margins_name='Total'))
# [DOC] "Canceled" and "No-Show" map 1:1 with is_canceled=1 => direct leakage

=== reservation_status x is_canceled ===


is_canceled             0      1   Total
reservation_status                      
Canceled                0  43017   43017
Check-Out           75166      0   75166
No-Show                 0   1207    1207
Total               75166  44224  119390


In [8]:
# ── Task 1 | Step 7: Distributions of key numeric variables ──────────────────
num_cols = ['lead_time', 'adr', 'days_in_waiting_list',
            'previous_cancellations', 'total_of_special_requests', 'total_nights']

print('=== Numeric summary statistics ===')
print(df[num_cols].describe().T.to_string())

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flatten(), num_cols):
    clip_val = df[col].quantile(0.99)
    df[col].clip(upper=clip_val).hist(bins=50, ax=ax, color='steelblue', edgecolor='none')
    ax.set_title(col)
    ax.set_xlabel('Value (clipped at 99th pct)')
    ax.set_ylabel('Count')
plt.suptitle('Key Numeric Feature Distributions', fontsize=12)
plt.tight_layout()
plt.savefig('task1_numeric_distributions.png', dpi=80)
plt.show()
print('Figure saved.')

=== Numeric summary statistics ===
                               count    mean     std    min    25%    50%     75%      max
lead_time                 119390.000 104.011 106.863  0.000 18.000 69.000 160.000  737.000
adr                       119390.000 101.831  50.536 -6.380 69.290 94.575 126.000 5400.000
days_in_waiting_list      119390.000   2.321  17.595  0.000  0.000  0.000   0.000  391.000
previous_cancellations    119390.000   0.087   0.844  0.000  0.000  0.000   0.000   26.000
total_of_special_requests 119390.000   0.571   0.793  0.000  0.000  0.000   1.000    5.000
total_nights              119390.000   3.428   2.557  0.000  2.000  3.000   4.000   69.000


Figure saved.


In [9]:
# ── Task 1 | Step 8: Cancellation rate by key categorical variables ───────────
cat_cols = ['hotel', 'market_segment', 'deposit_type', 'customer_type', 'distribution_channel']

fig, axes = plt.subplots(1, len(cat_cols), figsize=(18, 4))
for ax, col in zip(axes, cat_cols):
    rates = df.groupby(col)['is_canceled'].mean().sort_values(ascending=False)
    rates.plot(kind='bar', ax=ax, color='tomato', edgecolor='none')
    ax.set_title(f'Cancel rate\nby {col}', fontsize=9)
    ax.set_ylabel('Cancellation rate')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
    ax.set_ylim(0, 1)
plt.suptitle('Cancellation Rate by Categorical Variable', fontsize=11)
plt.tight_layout()
plt.savefig('task1_cancel_by_category.png', dpi=80)
plt.show()
print('Figure saved.')

Figure saved.


### 4. Verification

**Schema match:** The CSV has 32 columns against 31 documented variables; the extra column `hotel` is the hotel-type label added when the two per-hotel files were merged — consistent with the documentation.

**Missing values:**
- `children`: 4 true NaN rows (< 0.01%) — trivially imputable to 0 given the distribution.
- `country`: ~488 NaN rows (~0.4%) — imputable as "Unknown".
- `agent` / `company`: "NULL" strings are *documented as intentional*, meaning no agent or company was involved. They must **not** be treated as missing.

**Data quality issues confirmed:**
- Negative `adr`: present in data, consistent with H1 documentation (min = −6.38). Treat as erroneous; clip to 0.
- Extreme `adr` (> 1000): a small number of rows; likely data entry errors. Cap at a defensible upper bound in preprocessing.
- Zero-night bookings: present in data. Ambiguous — could be same-day events or errors. Retain and document.
- `adults` = 0 and `adults` > 10: both occur. Zero adults with children/babies is plausible; adults = 55 is not.
- Meal and deposit categorical levels match documentation exactly.

**Leakage confirmed:**
- `reservation_status` cross-tab shows a perfect 1:1 mapping with `is_canceled`. "Canceled" and "No-Show" statuses exclusively correspond to `is_canceled = 1`. This is definitive leakage.
- `reservation_status_date` must be dropped alongside it.
- `deposit_type`, `assigned_room_type`, and `booking_changes` carry partial post-outcome information per documentation; handled with caution in the preprocessing plan.

### 5. Revised Final Answer

#### Dataset schema understanding
119,390 bookings from two Portuguese hotels: H1 (Resort, Algarve, 40,060 rows) and H2 (City, Lisbon, 79,330 rows). Arrivals span July 2015 – August 2017. Each row is one booking. The target is `is_canceled` (binary). Data was extracted from the hotels' PMS systems with variable values captured as of the **day prior to arrival**.

#### Variable interpretation (from documentation)
All 32 columns are accounted for. Key interpretive notes grounded in documentation:
- *[DOC]* `agent` = "NULL" and `company` = "NULL" mean "no agent/company involved" — not missing values.
- *[DOC]* `deposit_type` is computed retroactively including payments up to the cancellation date — partial leakage risk.
- *[DOC]* `booking_changes` is counted up to check-in or cancellation — partial leakage risk.
- *[DOC]* `country` may reflect check-in information, not booking-time information.
- *[DOC]* `assigned_room_type` is determined at check-in, not booking time.

#### Missing value assessment
| Column | True NaN | Action |
|--------|----------|--------|
| `children` | 4 (0.003%) | Impute with 0 |
| `country` | ~488 (0.4%) | Impute with "Unknown" |
| `agent` | 0 (has "NULL" strings — not missing) | Encode as binary flag |
| `company` | 0 (has "NULL" strings — not missing) | Encode as binary flag |

#### Data quality issues detected
| Issue | Count | Action |
|-------|-------|--------|
| Duplicate rows | see output | Drop if confirmed exact duplicates |
| Negative `adr` | see output | Clip to 0 |
| `adr` > 1000 | see output | Cap at 99th percentile |
| Zero-night bookings | see output | Flag; retain but document |
| `adults` = 0 | see output | Retain if children/babies > 0; else flag |
| `adults` > 10 | see output | Investigate; likely outliers |

#### Leakage and methodological risk flags
| Variable | Risk level | Reason |
|----------|-----------|--------|
| `reservation_status` | **Critical — drop** | Direct post-outcome label (1:1 with target) |
| `reservation_status_date` | **Critical — drop** | Post-outcome timestamp |
| `deposit_type` | **High — use with caution** | *[DOC]* Computed from transactions up to cancellation date |
| `assigned_room_type` | **Medium — use with caution** | *[DOC]* Determined at check-in, not booking time |
| `booking_changes` | **Medium — use with caution** | *[DOC]* Counted up to check-in or cancellation |
| `country` | **Low — note** | *[DOC]* May not be confirmed until check-in |

#### Recommended preprocessing strategy
1. **Drop** `reservation_status` and `reservation_status_date` before any modelling.
2. **Retain but flag** `deposit_type`, `assigned_room_type`, `booking_changes` — document the risk; consider sensitivity experiments with and without them.
3. **Impute** `children` NaN → 0; `country` NaN → "Unknown".
4. **Encode** `agent`/`company` as binary flags (`has_agent`, `has_company`) per documentation intent.
5. **Clip** `adr` < 0 → 0; cap extreme high values at the 99th percentile.
6. **Retain** zero-night bookings for now; document as an uncertainty.
7. **Keep** `hotel` as a feature — the two hotels have meaningfully different cancellation rates.
8. All encoding and imputation must be fit on **training data only** to prevent leakage via the test set.

#### Uncertainties
- Whether `deposit_type` and `booking_changes` constitute true leakage depends on the deployment scenario (predictions made at booking creation vs. day-prior to arrival). This must be clarified before finalising the feature set.
- Duplicate rows may represent legitimate re-bookings; inspect all fields before dropping.
- `country` usability depends on whether the declared nationality is available at booking time.

---
## Task 2

---
## Task 3

---
## Task 4

---
## Task 5

---
## Task 6

---
## Task 7